In [0]:
%pip install soda-core-spark-df

In [0]:
# =============================================================================
# F1-Pulse | Data Quality — Silver Layer
# Notebook: 03_Silver_Quality
# Author:   Jafar891
# Updated:  2026
#
# Runs Soda checks against Silver Delta tables.
# Raises RuntimeError on failure to halt the Databricks job.
# =============================================================================

import logging

from soda.scan import Scan

from config.config import CATALOG, SILVER, MIN_LAP_DURATION_S

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.dq.silver")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
REPO_ROOT      = "/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse"
CHECKS_FILE    = f"{REPO_ROOT}/soda/checks_silver.yml"

# ---------------------------------------------------------------------------
# Load Silver tables and register as temp views
# Soda requires temp views — dots in table names are not allowed in view names
# ---------------------------------------------------------------------------
log.info("Registering Silver temp views …")
spark.table(f"{CATALOG}.{SILVER}.cleaned_sessions").createOrReplaceTempView("silver_cleaned_sessions")
spark.table(f"{CATALOG}.{SILVER}.enriched_laps").createOrReplaceTempView("silver_enriched_laps")

# ---------------------------------------------------------------------------
# Load and patch checks YAML
# Patches applied:
#   - Table names → temp view names (dots replaced with underscores)
#   - MIN_LAP_DURATION_S injected from config (single source of truth)
# ---------------------------------------------------------------------------
log.info(f"Loading checks from {CHECKS_FILE} …")
with open(CHECKS_FILE, "r") as f:
    yaml_content = f.read()

yaml_content = yaml_content.replace(
    "checks for silver.cleaned_sessions",
    "checks for silver_cleaned_sessions",
)
yaml_content = yaml_content.replace(
    "checks for silver.enriched_laps",
    "checks for silver_enriched_laps",
)

# Inject configured minimum lap duration into the specific threshold checks
yaml_content = yaml_content.replace(
    "valid min: 60",
    f"valid min: {MIN_LAP_DURATION_S}",
)
yaml_content = yaml_content.replace(
    f"min(lap_duration) >= 60",
    f"min(lap_duration) >= {MIN_LAP_DURATION_S}",
)

# ---------------------------------------------------------------------------
# Run Soda scan
# ---------------------------------------------------------------------------
log.info("Running Soda scan …")
scan = Scan()
scan.set_data_source_name("spark_df")
scan.add_spark_session(spark)
scan.add_sodacl_yaml_str(yaml_content)

exit_code = scan.execute()
log.info(scan.get_logs_text())

# ---------------------------------------------------------------------------
# Halt pipeline on failure — Databricks job marks task as FAILED
# ---------------------------------------------------------------------------
if exit_code != 0:
    raise RuntimeError("Silver DQ checks failed — halting pipeline.")

log.info("✅ Silver DQ checks passed.")